# Notebook 1 – Baseline: ResNet-18 trên CIFAR-10

**Đề tài 2.21 – Nghiên cứu và phòng thủ trước tấn công đối kháng lên mô hình phân loại ảnh**

Mục tiêu: huấn luyện mô hình ResNet-18 (bản cho CIFAR) trên dữ liệu sạch, làm mốc so sánh cho phần tấn công (Notebook 2) và phòng thủ (Notebook 3).

> Lưu ý quan trọng: ảnh đưa vào mô hình luôn nằm trong khoảng **[0, 1]**. Bước chuẩn hoá mean/std được đặt **bên trong mô hình** (lớp `Normalize` đầu mạng), nhờ đó epsilon của tấn công (vd. 8/255) có ý nghĩa đúng trên thang pixel.

## 0. Setup môi trường

- **Colab**: mount Google Drive (checkpoint lưu ở `MyDrive/AdvML_Project/checkpoints` để không mất khi runtime bị ngắt), clone repo vào `/content` nếu chưa có (có rồi thì `git pull`), rồi thêm repo vào `sys.path`.
- **Local**: thêm thư mục gốc repo (thư mục cha của `notebooks/`) vào `sys.path`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Eastern2k4/adversarial-attack-defense.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/AdvML_Project", exist_ok=True)

    REPO_DIR = Path("/content/adversarial-attack-defense")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    cwd = Path.cwd().resolve()
    REPO_DIR = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from src import config
from src.config import get_device
from src.data import get_loaders, get_eval_subset
from src.models import build_model
from src.utils import set_seed, evaluate, predict, train_one_epoch, save_checkpoint, load_checkpoint

print("Checkpoint dir:", config.CHECKPOINT_DIR)

## 1. Kiểm tra GPU, đặt seed, nạp dữ liệu

Trên Colab cần bật GPU: *Runtime → Change runtime type → T4 GPU*. Dữ liệu CIFAR-10 được tải tự động về thư mục `data/` (đã nằm trong `.gitignore`).

In [ ]:
device = get_device()
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

set_seed(config.SEED)
train_loader, test_loader = get_loaders(batch_size=config.BATCH_SIZE)
print(f"Train: {len(train_loader.dataset)} ảnh | Test: {len(test_loader.dataset)} ảnh")
x, y = next(iter(test_loader))
print("Batch:", tuple(x.shape), "| min/max pixel:", x.min().item(), x.max().item())

### Một số ảnh mẫu (tập test, chưa augmentation)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, label in zip(axes.flat, x[:16], y[:16]):
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(config.CLASSES[label], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Huấn luyện

- Optimizer: SGD (momentum, weight decay), lịch học `CosineAnnealingLR` trong `EPOCHS` epoch (các thông số lấy từ `src/config.py`).
- Sau **mỗi epoch** lưu checkpoint `baseline_last.pth` (model + optimizer + scheduler + epoch + best_acc + lịch sử) → nếu Colab bị ngắt, chỉ cần chạy lại notebook, cell này sẽ **tự resume**.
- Mô hình có accuracy test cao nhất được lưu riêng thành `baseline_resnet18.pth` (chỉ state_dict) – đây là file mà Notebook 2 và 3 sẽ dùng.

Thời gian tham khảo trên T4: khoảng 1 phút/epoch (~40 phút cho 40 epoch).

In [ ]:
model = build_model().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=config.LR,
                            momentum=config.MOMENTUM, weight_decay=config.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.EPOCHS)

LAST_CKPT = config.CHECKPOINT_DIR / "baseline_last.pth"
BEST_CKPT = config.CHECKPOINT_DIR / config.BASELINE_CKPT
config.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

start_epoch, best_acc = 0, 0.0
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": [], "lr": []}
if LAST_CKPT.exists():
    ckpt = load_checkpoint(LAST_CKPT, model, optimizer, scheduler, map_location=device)
    start_epoch, best_acc = ckpt["epoch"], ckpt["best_acc"]
    history = ckpt.get("history", history)
    print(f"Resume từ epoch {start_epoch}/{config.EPOCHS}, best_acc = {best_acc:.2f}%")
else:
    print("Không có checkpoint, huấn luyện từ đầu.")

In [ ]:
for epoch in range(start_epoch, config.EPOCHS):
    lr = optimizer.param_groups[0]["lr"]
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device,
                                            desc=f"Epoch {epoch + 1}/{config.EPOCHS}")
    test_loss, test_acc = evaluate(model, test_loader, device, criterion, return_loss=True)
    scheduler.step()

    for k, v in zip(history, (train_loss, train_acc, test_loss, test_acc, lr)):
        history[k].append(v)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), BEST_CKPT)
    save_checkpoint(LAST_CKPT, model, optimizer, scheduler,
                    epoch=epoch + 1, best_acc=best_acc, history=history)

    print(f"Epoch {epoch + 1:2d}/{config.EPOCHS} | lr {lr:.4f} | "
          f"train loss {train_loss:.4f} acc {train_acc:.2f}% | "
          f"test loss {test_loss:.4f} acc {test_acc:.2f}% | best {best_acc:.2f}%")

print("Hoàn tất. Best test accuracy:", f"{best_acc:.2f}%")

## 3. Biểu đồ loss / accuracy theo epoch

Hình được lưu vào `results/baseline_training_curves.png` để đưa vào báo cáo.

In [ ]:
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
epochs = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, history["train_loss"], label="train")
ax1.plot(epochs, history["test_loss"], label="test")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss"); ax1.legend(); ax1.grid(alpha=.3)
ax2.plot(epochs, history["train_acc"], label="train")
ax2.plot(epochs, history["test_acc"], label="test")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)"); ax2.set_title("Accuracy"); ax2.legend(); ax2.grid(alpha=.3)
plt.tight_layout()
plt.savefig(config.RESULTS_DIR / "baseline_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Đánh giá trên tập test sạch

Nạp lại mô hình tốt nhất (`baseline_resnet18.pth`), tính accuracy trên toàn bộ 10.000 ảnh test và accuracy theo từng lớp. Kết quả lưu ra `results/baseline_metrics.json`.

In [ ]:
best_model = build_model().to(device)
load_checkpoint(BEST_CKPT, best_model, map_location=device)

preds, labels = predict(best_model, test_loader, device)
clean_acc = 100.0 * (preds == labels).float().mean().item()
per_class = {
    name: 100.0 * (preds[labels == i] == i).float().mean().item()
    for i, name in enumerate(config.CLASSES)
}

print(f"Clean test accuracy: {clean_acc:.2f}%\n")
for name, acc in per_class.items():
    print(f"  {name:<12s} {acc:6.2f}%")

metrics = {
    "model": "ResNet-18 (CIFAR)",
    "clean_test_accuracy": round(clean_acc, 2),
    "per_class_accuracy": {k: round(v, 2) for k, v in per_class.items()},
    "epochs_trained": len(history["train_loss"]),
    "best_epoch": int(np.argmax(history["test_acc"])) + 1 if history["test_acc"] else None,
    "hyperparameters": {
        "seed": config.SEED, "batch_size": config.BATCH_SIZE, "epochs": config.EPOCHS,
        "lr": config.LR, "momentum": config.MOMENTUM, "weight_decay": config.WEIGHT_DECAY,
        "scheduler": "CosineAnnealingLR",
    },
    "history": history,
}
with open(config.RESULTS_DIR / "baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print("\nĐã lưu", config.RESULTS_DIR / "baseline_metrics.json")

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(per_class.keys(), per_class.values())
plt.axhline(clean_acc, color="red", ls="--", label=f"Trung bình {clean_acc:.2f}%")
plt.ylabel("Accuracy (%)"); plt.title("Accuracy theo từng lớp – baseline")
plt.xticks(rotation=30); plt.ylim(0, 100); plt.legend()
plt.tight_layout()
plt.savefig(config.RESULTS_DIR / "baseline_per_class.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Lưu kết quả

- `results/` nằm trong repo trên Colab (`/content/adversarial-attack-defense/results`) và **sẽ mất khi runtime bị ngắt** → tải các file PNG/JSON về hoặc commit chúng lên GitHub.
- Trọng số `baseline_resnet18.pth` nằm trên Drive (`MyDrive/AdvML_Project/checkpoints`). Không commit file `.pth` lên GitHub; chia sẻ thư mục Drive cho thành viên còn lại.